# 07. Detección de bordes

**Objetivo:** comprender qué es un borde y comparar Sobel, Laplaciano y Canny sobre una fotografía real.

In [ ]:
import numpy as np
import cv2

from filtrado_digital.io import cargar_imagen, a_grises, ruta_imagen_ejemplo
from filtrado_digital.visualizacion import comparar, mostrar_imagen
from filtrado_digital.bordes import (
    SOBEL_X,
    SOBEL_Y,
    sobel_manual,
    sobel_opencv,
    laplaciano_manual,
    laplaciano_opencv,
    canny_opencv,
)

## 1. ¿Qué es un borde?

Un **borde** es una zona donde la intensidad cambia de manera significativa entre píxeles cercanos. Estos cambios suelen coincidir con límites entre objetos, texturas o regiones diferentes.

El **gradiente** describe cuánto cambia la intensidad y en qué dirección. En una imagen podemos estimar cambios horizontales (`Gx`) y verticales (`Gy`). Una magnitud grande del gradiente sugiere la presencia de un borde.

In [ ]:
foto = cargar_imagen(ruta_imagen_ejemplo())
gris = a_grises(foto)
imagen = gris[200:392, 440:632]

# Un suavizado ligero reduce variaciones pequeñas antes de buscar bordes.
imagen_suave = cv2.GaussianBlur(imagen, (5, 5), 1.0)
comparar([imagen, imagen_suave], ["Original", "Suavizado previo"])

## 2. Sobel

Sobel utiliza dos kernels: uno responde principalmente a cambios horizontales de intensidad y otro a cambios verticales.

In [ ]:
print("Sobel X:\n", SOBEL_X)
print("\nSobel Y:\n", SOBEL_Y)

gx, gy, magnitud_manual = sobel_manual(imagen_suave)
_, _, magnitud_cv = sobel_opencv(imagen_suave)

comparar([magnitud_manual, magnitud_cv], ["Sobel manual", "Sobel OpenCV"])

## 3. Laplaciano

El **Laplaciano** responde a cambios rápidos de intensidad combinando información de distintas direcciones. Es sensible al ruido, por lo que suele aplicarse después de algún suavizado.

In [ ]:
lap_manual = laplaciano_manual(imagen_suave)
lap_cv = laplaciano_opencv(imagen_suave)
comparar([imagen_suave, lap_manual, lap_cv], ["Suavizada", "Laplaciano manual", "Laplaciano OpenCV"])

## 4. Canny

Canny es un detector de bordes compuesto por varias etapas. De forma simplificada:

1. reduce ruido;
2. calcula gradientes;
3. conserva máximos locales para adelgazar bordes;
4. utiliza dos umbrales para distinguir respuestas fuertes y débiles;
5. conecta bordes débiles que están relacionados con bordes fuertes.

No implementaremos todas estas etapas desde cero en este curso; utilizaremos la función de OpenCV y estudiaremos el efecto de sus umbrales.

In [ ]:
canny_sensible = canny_opencv(imagen_suave, 40, 100)
canny_selectivo = canny_opencv(imagen_suave, 100, 200)
comparar(
    [imagen_suave, canny_sensible, canny_selectivo],
    ["Imagen", "Canny 40/100", "Canny 100/200"],
)

## 5. Comparar métodos

Sobel y Laplaciano producen respuestas de intensidad que muestran dónde hay cambios. Canny devuelve directamente una imagen binaria de bordes y agrega etapas para reducir respuestas débiles o duplicadas.

In [ ]:
comparar(
    [magnitud_cv, lap_cv, canny_selectivo],
    ["Sobel", "Laplaciano", "Canny"],
)

## Conclusiones

- Los bordes corresponden a cambios importantes de intensidad.
- Sobel estima gradientes horizontales y verticales.
- El Laplaciano responde a cambios rápidos en varias direcciones y es sensible al ruido.
- Canny combina suavizado, gradiente y umbrales para producir bordes delgados y conectados.
- Los parámetros cambian qué detalles se consideran bordes relevantes.